# Concept map: checks behind DATA_ISSUES.md

Each section reruns the check that supports an issue in `DATA_ISSUES.md` (IDs in brackets). These are slow (some read millions of rows) and only need rerunning if a supplier changes the data. Counts and ranges only.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from datetime import date
from pathlib import Path
import pandas as pd
open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))
import concepts as C

In [ ]:
open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")

assembled_dir = open_picu_dir / "derived" / "assembled"
reports_dir = open_picu_dir / "reports"
concepts_dir = open_picu_dir / "derived" / "concepts"
detail = pd.read_csv(Path(concepts_dir) / "concept_detail.csv")

## A. Fluids [G9 fluids, S3]
### A1. SMH: is `hourTotal` the change in `cumTotal`?
Per intervention ID: gap between rows, share of resets, and among the others the share where diff(`cumTotal`) = `hourTotal`. Result: 99 to 100% match, so `hourTotal` is hourly and `cumTotal` a running total resetting about daily.

In [ ]:
C.io_diff_check(assembled_dir)

### A2. SMH: do the intake components add up to the total?
IV infusion (1851) + PO/NG (1912) + blood (2052) + colloids (2047) + drug volume (2101) = total intake. Result: equal in 99.4% of stay-hours and never above it. The IV-only definition (no blood products) is 1851 + 2047 + 2101.

In [ ]:
C.io_reconcile(assembled_dir, total_ids=["2045", "113860"], component_ids=["1851", "1912", "2052", "2047", "2101"])

### A3. GOSH balance files: row intervals and running totals
`start_datetime` = `end_datetime` and rows are about hourly. "Total Fluids In/Out (24hrs)" are running totals that reset at 07:00. "LOS Net Body Balance" is a stay-long balance, so differencing gives hourly net balance with no resets.

In [ ]:
site, bal = "GOSH", C._BAL
C.interval_check(assembled_dir, site, bal + "Total_fluids", "shortLabel", "start_datetime", "end_datetime")
C.cum_check(assembled_dir, site, bal + "Total_fluids", "shortLabel",
            ["Total Fluids In (24hrs)", "Total Fluids Out (24hrs)"], "start_datetime", "cumTotal")
C.cum_check(assembled_dir, site, bal + "Net_Bal", "shortLabel", ["LOS Net Body Balance"], "start_datetime", "cumTotal")

### A4. GOSH: do the balance files cover the Epic years?
Years in which each label appears. **This decides whether GOSH hourly fluids can come from differencing across all years or must use `caboodle_fluid_balance` after the Epic switch.**

In [ ]:
C.names_by_year(assembled_dir, "GOSH", C._BAL + "Total_fluids", "shortLabel", "start_datetime")
C.names_by_year(assembled_dir, "GOSH", C._BAL + "Net_Bal", "shortLabel", "start_datetime")

### A5. GOSH `caboodle_fluid_balance` (hourly, 2019-04 to 2022-12)
With the 07:00 day boundary the hourly sums equal the daily columns on about 93.8% (intake) and 92.2% (output) of stay-days. The source is sparse (many zero hours), so zeros are not recorded absences.

In [ ]:
C.caboodle_check(assembled_dir)

### A6. RBH: which `io` columns add up to total intake? [definition of `intake_iv`]
Treats the six `bolus_*` columns as one component. For stay-hours with a positive `Total_In_ex_dialysis`, shows the share of hours where the total equals the sum of each subset of components (missing counted as 0, 1 ml tolerance), best subsets first. It tells us whether the boluses are separate from `IV_Infusion`, and which columns the RBH total contains. The IV-only definition (no blood, no enteral) is then the matching subset without `Blood_In`, `Enteral_PO`, `PO_NG_Intake` and `TPN`, as with SMH (IV infusion, colloids and drug volume).

In [ ]:
C.rbh_io_reconcile(assembled_dir)

### A6b. RBH: does the intake total change meaning over time?
By year, for stay-hours with a positive total: the share where `Total_In_ex_dialysis` equals the sum of the best-fitting columns (`IV_Infusion`, `IV_Meds`, `Colloids`, `Blood_In`, `Enteral_PO`), the shares where the total is above and below that sum, and how often each column (and `PO_NG_Intake`, `TPN`) is non-zero. A step in any of these shows a change in how RBH recorded intake, as with the two SMH system versions.

In [ ]:
C.rbh_io_by_year(assembled_dir).T

## B. Units and value checks [G3, G4, G7, G8, S2, R3]
### B1. Placeholder zeros, text values, lost negatives
Per concept, site and source: percent non-numeric, exactly zero, and negative.

In [ ]:
names = [n for n in C.CONCEPTS if C.CONCEPTS[n]["group"] != "drugs"]
C.value_checks(assembled_dir, names)

### B2. Base excess sign (GOSH has no negative values) [G3]
Median base excess by pH band. In acidotic samples GOSH is positive (about +6.6) against about -4.7 at RBH, so the sign was lost at source.

In [ ]:
C.be_ph_check(assembled_dir, "RBH"); C.be_ph_check(assembled_dir, "GOSH")

### B2b. Can GOSH base excess be derived? [G3]
Base excess can be recomputed from pH, bicarbonate and haemoglobin (Siggaard-Andersen; Van Slyke form where haemoglobin is missing).

- **RBH and SMH** (recorded base excess keeps its sign): how closely does the derived value match the recorded one? If the median difference is near zero and most samples are within 1 to 2 mmol/L, the formula is fit to use.
- **GOSH** (sign lost): is the recorded value about the magnitude of the derived one (recorded minus |derived| near zero)? If so, that confirms the sign was stripped at source, and that GOSH bicarbonate and haemoglobin are usable.

Counts, medians and shares only. Haemoglobin in g/L is converted to g/dL.

In [ ]:
for site in ("RBH", "SMH", "GOSH"):
    C.be_derive_check(assembled_dir, site)

### B3. GOSH diastolic blood pressure copies systolic [G4]
About 37% of GOSH readings have diastolic = systolic exactly (RBH 0.0 to 0.4%). Rule: diastolic = systolic is set to missing.

In [ ]:
C.bp_order_check(assembled_dir, "GOSH", kind="NIBP"); C.bp_order_check(assembled_dir, "GOSH", kind="IBP")
C.bp_order_check(assembled_dir, "RBH", kind="NIBP")

### B4. Value formats and haematocrit labels [G6, G7]
Text formats (tidal volume "9 ml", capillary refill categories, airway device codes) and haematocrit by label (percent vs fraction).

In [ ]:
for c in ("tidal_volume", "crt", "airway_device"):
    print(c); print(C.value_shapes(assembled_dir, c, "GOSH"))

## C. I:E ratio [S4, G9]
SMH IDs 4185 and 4186 hold two rows per time (the smaller is 1 in 99.7 to 99.8%, never equal). The ratio x is the larger.

In [ ]:
C.ie_pair_check(assembled_dir)
C.list_values(assembled_dir, "GOSH", C._VENT + "merged_IE_ratios", "source")

## D. Drug sources by year [G1, G2, R4]
Where each GOSH drug table starts and stops (the Epic switch about 2019-04), and the RBH `sedation` / `sedation_old` overlap.

In [ ]:
import pyarrow.parquet as pq

def _timecol(site, table):
    cols = pq.read_schema(Path(assembled_dir) / site / f"{table}.parquet").names
    cand = [c for c in cols if "time" in c.lower() or "date" in c.lower()]
    return cand[0] if cand else None

gosh_drugs = {C._CI: "shortLabel", C._INTRV: "shortLabel", C._ALL: "shortLabel", C._MEDS: "drug_name"}
for table, label in gosh_drugs.items():
    t = _timecol("GOSH", table)
    print(f"\n=== GOSH {table}  (label: {label}, time: {t}) ===")
    if t:
        print(C.names_by_year(assembled_dir, "GOSH", table, label, t, top=12))

### RBH sedation files
`sedation` and `sedation_old` overlap in time, so drug presence is the union of the two (R4). The table shows which years each file covers.

In [ ]:
for table in ("ai_picu_sedation", "ai_picu_sedation_old"):
    t = _timecol("RBH", table)
    print(f"\n=== RBH {table}  (label: Drug, time: {t}) ===")
    if t:
        print(C.names_by_year(assembled_dir, "RBH", table, "Drug", t, top=12))